# 03 — Geometric Preprocessing (Real Ground-Control-Point Georeferencing)

Your OHRC and TMC-2 downloads include a `geometry/*.csv` per product with
columns **Longitude, Latitude, Pixel, Scan** — a sparse grid of real ground
control points (GCPs) tying image (sample, line) coordinates to actual lunar
lon/lat. This is much better than guessing an approximate overlap crop, so
this notebook uses it directly.

**IIRS has no such file in your download** — its raw product isn't
georeferenced this way. We leave IIRS in pixel space for now; it gets tied
into the same reference frame indirectly once it's feature-matched against
TMC-2 in Notebook 05/06 (matched-point pairs give you IIRS pixel ↔ TMC-2
pixel ↔ real lon/lat, transitively).

**Output:**
```
26166_DATA/processed/geocoded/{ohrc,tmc2}/<product>.tif   (real CRS attached)
26166_DATA/processed/manifest_geocoded.csv                 (includes computed real GSD)
```


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, csv, math
import numpy as np
import rasterio
from rasterio.transform import from_gcps
from rasterio.control import GroundControlPoint
import matplotlib.pyplot as plt

DRIVE_ROOT = '/content/drive/MyDrive/26166_DATA'
RAW_ROOT   = f'{DRIVE_ROOT}/raw'
IN_ROOT    = f'{DRIVE_ROOT}/processed/radiometric'
OUT_ROOT   = f'{DRIVE_ROOT}/processed/geocoded'
os.makedirs(OUT_ROOT, exist_ok=True)

with open(f'{DRIVE_ROOT}/processed/manifest_radiometric.csv') as f:
    rows = list(csv.DictReader(f))

# Simple spherical Moon CRS (mean radius 1737.4 km) — avoids depending on
# whether this Colab's PROJ database includes IAU lunar authority codes.
MOON_PROJ4 = '+proj=longlat +a=1737400 +b=1737400 +no_defs'
MOON_RADIUS_M = 1737400.0


## 1. Find and parse each product's geometry CSV

The geometry file lives alongside the image, under `geometry/<level>/<date>/`
rather than `data/<level>/<date>/` — same product timestamp, different
subfolder. We locate it by matching the acquisition-time string in the filename.


In [ ]:
from pathlib import Path

def find_geometry_csv(instrument, product_stem):
    """product_stem looks like 'ch2_ohr_ncp_20220914T0835371412_d_img_d32'.
    The geometry CSV for the same acquisition is named with '_g_grd_' instead
    of '_d_img_' and lives under geometry/ instead of data/."""
    geom_root = Path(f'{RAW_ROOT}/{instrument}/geometry')
    if not geom_root.exists():
        return None
    # Extract the shared timestamp token, e.g. '20220914T0835371412'
    parts = product_stem.split('_')
    timestamp_token = next((p for p in parts if 'T' in p and p[:1].isdigit()), None)
    if timestamp_token is None:
        return None
    matches = [p for p in geom_root.rglob('*.csv') if timestamp_token in p.name]
    return matches[0] if matches else None

def load_gcps(csv_path, sample_stride=1):
    gcps = []
    with open(csv_path) as f:
        reader = csv.DictReader(f)
        for i, row in enumerate(reader):
            if i % sample_stride != 0:
                continue
            lon, lat = float(row['Longitude']), float(row['Latitude'])
            col, line = float(row['Pixel']), float(row['Scan'])
            gcps.append(GroundControlPoint(row=line, col=col, x=lon, y=lat))
    return gcps


## 2. Compute real ground sampling distance (GSD) from the GCPs

Rather than trusting a nominal literature value (0.3 m / 5 m), derive the
*actual* per-product GSD from the great-circle distance between two nearby
GCPs divided by their pixel spacing — useful both for sanity-checking and for
feeding an accurate value into Notebook 04's scale-space step.


In [ ]:
def great_circle_distance_m(lat1, lon1, lat2, lon2, radius=MOON_RADIUS_M):
    lat1, lon1, lat2, lon2 = map(math.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = math.sin(dlat/2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon/2)**2
    return 2 * radius * math.asin(math.sqrt(a))

def estimate_gsd_from_gcps(gcps):
    """Uses the first two GCPs that share the same Scan/line value (i.e. are
    horizontally adjacent along-track) to estimate across-track GSD."""
    by_row = {}
    for g in gcps:
        by_row.setdefault(g.row, []).append(g)
    for row_val, pts in by_row.items():
        if len(pts) >= 2:
            pts = sorted(pts, key=lambda g: g.col)
            p1, p2 = pts[0], pts[1]
            dist_m = great_circle_distance_m(p1.y, p1.x, p2.y, p2.x)
            pixel_spacing = abs(p2.col - p1.col)
            if pixel_spacing > 0:
                return dist_m / pixel_spacing
    return None


## 3. Attach real georeferencing to OHRC & TMC-2

`from_gcps()` fits a single affine transform through all control points —
appropriate for narrow-swath pushbroom sensors like these, where within-scene
distortion is small relative to a simple affine model. This does **not**
resample pixels (fast, no interpolation loss); it just attaches an accurate
transform + CRS to the existing pixel grid. If you later see visible warp
error, upgrade to a true polynomial/TPS resampling (`gdalwarp -tps`) as a
stretch goal — noted at the bottom of this notebook.


In [ ]:
geocoded_rows = []

for r in rows:
    if r['instrument'] not in ('ohrc', 'tmc2'):
        continue   # IIRS has no geometry CSV in this download — skipped here, see notebook intro

    geom_csv = find_geometry_csv(r['instrument'], r['product'])
    if geom_csv is None:
        print(f"  [no geometry CSV found] {r['product']} — leaving ungeoreferenced")
        continue

    gcps = load_gcps(geom_csv)
    if len(gcps) < 3:
        print(f"  [too few GCPs] {r['product']}: only {len(gcps)} points")
        continue

    transform = from_gcps(gcps)
    gsd_m = estimate_gsd_from_gcps(gcps)

    with rasterio.open(r['radiometric_path']) as src:
        arr = src.read()
        profile = src.profile.copy()

    profile.update({'crs': MOON_PROJ4, 'transform': transform})

    out_dir = f"{OUT_ROOT}/{r['instrument']}"
    os.makedirs(out_dir, exist_ok=True)
    out_path = f"{out_dir}/{r['product']}.tif"
    with rasterio.open(out_path, 'w', **profile) as dst:
        dst.write(arr)

    geocoded_rows.append({**r, 'geocoded_path': out_path, 'n_gcps': len(gcps),
                           'estimated_gsd_m': gsd_m, 'geometry_csv': str(geom_csv)})
    print(f"{r['product']}: attached {len(gcps)} GCPs, estimated GSD = {gsd_m:.3f} m/px"
          if gsd_m else f"{r['product']}: attached {len(gcps)} GCPs, GSD estimation failed")


## 4. Visual check — plot the OHRC and TMC-2 footprints on a lon/lat grid

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
colors = {'ohrc': 'red', 'tmc2': 'blue'}
for r in geocoded_rows:
    with rasterio.open(r['geocoded_path']) as src:
        b = src.bounds
    from matplotlib.patches import Rectangle
    ax.add_patch(Rectangle((b.left, b.bottom), b.right - b.left, b.top - b.bottom,
                            fill=False, edgecolor=colors[r['instrument']], linewidth=2,
                            label=r['instrument']))
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
ax.autoscale(); ax.set_aspect('equal')
handles, labels = ax.get_legend_handles_labels()
by_label = dict(zip(labels, handles))
ax.legend(by_label.values(), by_label.keys())
plt.title('OHRC vs TMC-2 footprints (real georeferencing)')
plt.grid(True, alpha=0.3)
plt.show()

print('If these two rectangles do NOT overlap, your OHRC and TMC-2 downloads')
print('are of different ground sites despite both coming from the CHMAP layers')
print('view — double check before investing time in matching.')


In [ ]:
out_manifest = f'{DRIVE_ROOT}/processed/manifest_geocoded.csv'
with open(out_manifest, 'w', newline='') as f:
    fieldnames = list(geocoded_rows[0].keys())
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(geocoded_rows)
print('Saved', out_manifest)


---
### Notes / stretch goals
- **Check the footprint overlap plot in Section 4 first**, before anything
  else — your OHRC (2022-09-14) and TMC-2 (2019-11-25) acquisitions are three
  years apart; confirming they cover the same ground is more important than
  any code correctness here.
- If the simple affine fit shows visible internal warp (check by re-plotting
  a few individual GCPs against the fitted transform's predicted position),
  upgrade to a real resampled warp:
  `gdalwarp -r bilinear -tps -a_srs "+proj=longlat +a=1737400 +b=1737400" src.tif dst.tif`
  called via `subprocess`, using the same GCPs written into the source file
  with `gdal_translate -a_srs ... -gcp ...` first.
- IIRS: once you have OHRC/TMC-2↔IIRS pixel matches from Notebook 05, you can
  back-compute approximate lon/lat for IIRS pixels via the already-georeferenced
  TMC-2 image — a legitimate way to "inherit" georeferencing transitively.
- Next notebook: **04_scale_space.ipynb**
